# **Image Classification with Hugging Face Transformers and `Keras`**

The Vision Transformer (ViT) is a deep learning model that adapts the Transformer architecture, originally developed for NLP, to handle image data. ViT splits an image into fixed-size patches, linearly embeds them, and processes them as a sequence of tokens similar to words in text. It captures long-range dependencies and global context effectively, often outperforming convolutional neural networks (CNNs) in various vision tasks when pre-trained on large datasets. ViT models are particularly known for their scalability and high accuracy in image classification.

Reference : https://www.philschmid.de/image-classification-huggingface-transformers-keras

Paper: https://arxiv.org/abs/2010.11929

Official repo (in JAX): https://github.com/google-research/vision_transformer



## **Load Dataset**

In [1]:
!pip install datasets==2.21.0 -q
!pip install tensorflow==2.15.0 -q
!pip install transformers==4.41.2 -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-decision-forests 1.9.1 requires tensorflow~=2.16.1, but you have tensorflow 2.15.0 which is incompatible.
tensorflow-serving-api 2.16.1 requires tensorflow<3,>=2.16.1, but you have tensorflow 2.15.0 which is incompatible.
tensorflow-text 2.16.1 requires tensorflow<2.17,>=2.16.1; platform_machine != "arm64" or platform_system != "Darwin", but you have tensorflow 2.15.0 which is incompatible.
tensorstore 0.1.66 requires ml-dtypes>=0.3.1, but you have ml-dtypes 0.2.0 which is incompatible.
tf-keras 2.16.0 requires tensorflow<2.17,>=2.16, but you have tensorflow 2.15.0 which is incompatible.


### Load dataset use datasets

In [89]:
from datasets import load_dataset
ds2 = load_dataset('/kaggle/input/glasses-classification-dataset/train')
ds2

Resolving data files:   0%|          | 0/104 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 104
    })
})

In [91]:
ds=ds2["train"]
ds = ds.rename_column("image", "img")
ds

Dataset({
    features: ['img', 'label'],
    num_rows: 104
})

### Load dataset use python script

In [2]:
import os
import datasets
 
def create_image_folder_dataset(root_path):
  """creates `Dataset` from image folder structure"""
 
  # get class names by folders names
  _CLASS_NAMES= os.listdir(root_path)
  # defines `datasets` features`
  features=datasets.Features({
                      "img": datasets.Image(),
                      "label": datasets.features.ClassLabel(names=_CLASS_NAMES),
                  })
  # temp list holding datapoints for creation
  img_data_files=[]
  label_data_files=[]
  # load images into list for creation
  for img_class in os.listdir(root_path):
    for img in os.listdir(os.path.join(root_path,img_class)):
      path_=os.path.join(root_path,img_class,img)
      img_data_files.append(path_)
      label_data_files.append(img_class)
    
  # create dataset
  ds = datasets.Dataset.from_dict({"img":img_data_files,"label":label_data_files},features=features)
  return ds
 

In [3]:
ds = create_image_folder_dataset("/kaggle/input/glasses-classification-dataset/train")
ds

Dataset({
    features: ['img', 'label'],
    num_rows: 104
})

In [4]:
.features["label"]

ClassLabel(names=['glasses', 'noglasses'], id=None)

## **Pre-processing**

In [5]:
from transformers import ViTFeatureExtractor

model_id = "google/vit-base-patch16-224-in21k"
feature_extractor = ViTFeatureExtractor.from_pretrained(model_id)

feature_extractor

2024-10-26 11:02:54.514432: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-10-26 11:02:54.514496: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-10-26 11:02:54.515988: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

/opt/conda/lib/python3.10/site-packages/transformers/models/vit/feature_extraction_vit.py:28: FutureWarning: The class ViTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ViTImageProcessor instead.
  warnings.warn(


ViTFeatureExtractor {
  "_valid_processor_keys": [
    "images",
    "do_resize",
    "size",
    "resample",
    "do_rescale",
    "rescale_factor",
    "do_normalize",
    "image_mean",
    "image_std",
    "return_tensors",
    "data_format",
    "input_data_format"
  ],
  "do_normalize": true,
  "do_rescale": true,
  "do_resize": true,
  "image_mean": [
    0.5,
    0.5,
    0.5
  ],
  "image_processor_type": "ViTFeatureExtractor",
  "image_std": [
    0.5,
    0.5,
    0.5
  ],
  "resample": 2,
  "rescale_factor": 0.00392156862745098,
  "size": {
    "height": 224,
    "width": 224
  }
}

In [7]:

from tensorflow import keras
from tensorflow.keras import layers
 
 

 
# learn more about data augmentation here: https://www.tensorflow.org/tutorials/images/data_augmentation
data_augmentation = keras.Sequential(
    [
        layers.Resizing(feature_extractor.size, feature_extractor.size),
        layers.Rescaling(1./255),
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(factor=0.02),
        layers.RandomZoom(
            height_factor=0.2, width_factor=0.2
        ),
    ],
    name="data_augmentation",
)
# use keras image data augementation processing
def augmentation(examples):
    # print(examples["img"])
    examples["pixel_values"] = [data_augmentation(image) for image in examples["img"]]
    return examples
 
 
# basic processing (only resizing)
def process(examples):
    examples.update(feature_extractor(examples['img'], ))
    return examples
 

 
 

In [8]:
# we are also renaming our label col to labels to use `.to_tf_dataset` later
ds = ds.rename_column("label", "labels")

In [9]:
processed_dataset = ds.map(process, batched=True)
processed_dataset
 
#augmenting dataset takes a lot of time
#processed_dataset = ds.map(augmentation, batched=True)
#processed_dataset

Map:   0%|          | 0/104 [00:00<?, ? examples/s]

Dataset({
    features: ['img', 'labels', 'pixel_values'],
    num_rows: 104
})

In [10]:
# test size will be 15% of train dataset
test_size=.15
 
processed_dataset = processed_dataset.shuffle().train_test_split(test_size=test_size)
train_ds = processed_dataset['train']
val_ds = processed_dataset['test']

## **Fine-tuning the model using Keras**

In [11]:
labels = ds.features["labels"].names
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = i
    id2label[i] = label
    
    
print("lebel2id :",label2id)
print("id2label :",id2label)

lebel2id : {'glasses': 0, 'noglasses': 1}
id2label : {0: 'glasses', 1: 'noglasses'}


In [12]:
len(labels )

2

In [21]:
from huggingface_hub import HfFolder
import tensorflow as tf

 
num_train_epochs = 10
train_batch_size = 16
eval_batch_size = 16
learning_rate = 3e-5
weight_decay_rate=0.01
num_warmup_steps=0
output_dir=model_id.split("/")[1]
hub_token = HfFolder.get_token() # or your token directly "hf_xxx"
hub_model_id = f'{model_id.split("/")[1]}-euroSat'
fp16=True
 
# Train in mixed-precision float16
# Comment this line out if you're using a GPU that will not benefit from this
if fp16:
  tf.keras.mixed_precision.set_global_policy("mixed_float16")
 

## **Converting the dataset to a tf.data.Dataset**

In [22]:
from transformers import DefaultDataCollator
 
# Data collator that will dynamically pad the inputs received, as well as the labels.
data_collator = DefaultDataCollator(return_tensors="tf")
 
# converting our train dataset to tf.data.Dataset
tf_train_dataset = train_ds.to_tf_dataset(
   columns=['pixel_values'],
   label_cols=["labels"],
   shuffle=True,
   batch_size=train_batch_size,
   collate_fn=data_collator)
 
# converting our test dataset to tf.data.Dataset
tf_eval_dataset = val_ds.to_tf_dataset(
   columns=['pixel_values'],
   label_cols=["labels"],
   shuffle=True,
   batch_size=eval_batch_size,
   collate_fn=data_collator)

/opt/conda/lib/python3.10/site-packages/datasets/arrow_dataset.py:410: FutureWarning: The output of `to_tf_dataset` will change when a passing single element list for `labels` or `columns` in the next datasets version. To return a tuple structure rather than dict, pass a single string.
Old behaviour: columns=['a'], labels=['labels'] -> (tf.Tensor, tf.Tensor)  
             : columns='a', labels='labels' -> (tf.Tensor, tf.Tensor)  
New behaviour: columns=['a'],labels=['labels'] -> ({'a': tf.Tensor}, {'labels': tf.Tensor})  
             : columns='a', labels='labels' -> (tf.Tensor, tf.Tensor) 
  warnings.warn(


## **Download the pre-trained transformer model and fine-tune it**

In [23]:
from transformers import TFViTForImageClassification, create_optimizer
import tensorflow as tf
 
# create optimizer wight weigh decay
num_train_steps = len(tf_train_dataset) * num_train_epochs
optimizer, lr_schedule = create_optimizer(
    init_lr=learning_rate,
    num_train_steps=num_train_steps,
    weight_decay_rate=weight_decay_rate,
    num_warmup_steps=num_warmup_steps,
)
 
# load pre-trained ViT model
model = TFViTForImageClassification.from_pretrained(
    model_id,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)
 
# define loss
loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
 
# define metrics
metrics=[
    tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
    tf.keras.metrics.SparseTopKCategoricalAccuracy(len(labels), name="top-3-accuracy"),
]
 


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFViTForImageClassification: ['pooler.dense.bias', 'pooler.dense.weight']
- This IS expected if you are initializing TFViTForImageClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFViTForImageClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFViTForImageClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


If you want to create you own classification head or if you want to add the augmentation/processing layer to your model, you can directly use the functional Keras API. Below you find an example on how you would create a classification head.

In [23]:
# alternatively create Image Classification model using Keras Layer and ViTModel
# here you can also add the processing layers of keras
 
import tensorflow as tf
from transformers import TFViTModel
 
base_model = TFViTModel.from_pretrained('google/vit-base-patch16-224-in21k')
 
 
# inputs
pixel_values = tf.keras.layers.Input(shape=(3,224,224), name='pixel_values', dtype='float32')
 
# model layer
@tf.function
def process_inputs(inputs):
    return base_model.vit(inputs)[0]
vit = tf.keras.layers.Lambda(process_inputs)(pixel_values)

#vit = base_model.vit(pixel_values)[0]
classifier = tf.keras.layers.Dense(len(labels), activation='softmax', name='outputs')(vit[:, 0, :])
 
# model
keras_model = tf.keras.Model(inputs=pixel_values, outputs=classifier)

All PyTorch model weights were used when initializing TFViTModel.

All the weights of TFViTModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFViTModel for predictions without further training.


In [33]:
from tensorflow.keras.optimizers import Adam

#optimizer = Adam(learning_rate=1e-5)


# compile model
model.compile(optimizer="adam",
              loss=loss,
              metrics=metrics,
            run_eagerly=True)

#keras_model.compile(optimizer=optimizer,
#              loss=loss,
 #             metrics=metrics,)

## **Callbacks**

In [34]:
from keras.callbacks import ModelCheckpoint , EarlyStopping

model_dir=os.path.join(output_dir,"model.keras")
# Define the ModelCheckpoint callback
checkpoint = ModelCheckpoint(model_dir, save_best_only=True, monitor='val_loss', verbose=1)

In [35]:
import os
from transformers.keras_callbacks import PushToHubCallback
from tensorflow.keras.callbacks import TensorBoard as TensorboardCallback, EarlyStopping
 
callbacks=[]
 
callbacks.append(TensorboardCallback(log_dir=os.path.join(output_dir,"logs")))
callbacks.append(EarlyStopping(monitor="val_accuracy",patience=5))
if hub_token:
  callbacks.append(PushToHubCallback(output_dir=output_dir,
                                     hub_model_id=hub_model_id,
                                     hub_token=hub_token))
 
 

In [36]:
train_results =model.fit(
    tf_train_dataset,
    validation_data=tf_eval_dataset,
    callbacks=[callbacks,checkpoint],
    epochs=num_train_epochs,
)

Epoch 1/10


I0000 00:00:1729941377.442617      30 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


6/6 [==============================] - ETA: 0s - loss: 0.6852 - accuracy: 0.9734 - top-3-accuracy: 1.0000
Epoch 1: val_loss improved from inf to 0.66375, saving model to vit-base-patch16-224-in21k/model.keras
6/6 [==============================] - 39s 2s/step - loss: 0.6852 - accuracy: 0.9734 - top-3-accuracy: 1.0000 - val_loss: 0.6637 - val_accuracy: 0.9629 - val_top-3-accuracy: 1.0000
Epoch 2/10
6/6 [==============================] - ETA: 0s - loss: 0.7192 - accuracy: 0.9514 - top-3-accuracy: 1.0000
Epoch 2: val_loss did not improve from 0.66375
6/6 [==============================] - 48s 940ms/step - loss: 0.7192 - accuracy: 0.9514 - top-3-accuracy: 1.0000 - val_loss: 0.7401 - val_accuracy: 0.9401 - val_top-3-accuracy: 1.0000
Epoch 3/10
6/6 [==============================] - ETA: 0s - loss: 0.7023 - accuracy: 0.9293 - top-3-accuracy: 1.0000
Epoch 3: val_loss did not improve from 0.66375
6/6 [==============================] - 9s 932ms/step - loss: 0.7023 - accuracy: 0.9293 - top-3-acc